# CC3 : SmartGrid - Ordonnancement de la production energetique sous incertitude

**Etude de cas interdisciplinaire** combinant programmation par contraintes (OR-Tools CP-SAT),
inference probabiliste (modele bayesien), optimisation multi-objectif et un jumeau numerique
de reseau electrique.


## Contexte metier

Un operateur de reseau electrique doit, pour chaque heure, decider quelles centrales activer
et a quel niveau de production (le *unit commitment* / dispatch) afin de satisfaire la demande
tout en minimisant le cout economique et les emissions de CO2. Ce probleme est rendu difficile
par :

- **l'incertitude** sur la production renouvelable (eolien, solaire) et sur la demande ;
- **les contraintes dures** : capacites min/max des centrales, temps de montee/descente,
  reserve tournante (securite n-1) ;
- **les objectifs multiples conflictuels** : cout, emissions, fiabilite.

Une seule technique ne suffit pas : un solveur deterministe ignore l'incertitude, un modele
probabiliste seul ne respecte pas les contraintes physiques. Cette etude de cas materialise
leur **composition ordonnee** : filtrer les dispatches impossibles (contraintes), modeliser
l'aleatoire (incertitude), puis optimiser (multi-objectif).


## Architecture en 4 couches

| Couche | Role | Technique | Partie |
|--------|------|-----------|--------|
| **Contraintes dures** | Filtrer les dispatches impossibles | OR-Tools CP-SAT | Partie 1 |
| **Incertitude** | Modeliser l'aleatoire renouvelable | Modele bayesien | Partie 2 |
| **Optimisation** | Choisir le meilleur compromis | Multi-objectif pondere | Partie 3 |
| **Decision** | Synthese + comparaison | Workflow + scoring | Partie 4 |

Ce notebook est une **fondation** : le jumeau numerique (couche 0) est operationnel ; les
implementations des 4 couches seront completees et executees dans les cycles suivants.


## 0. Installation et jumeau numerique

Le jumeau numerique represente un reseau electrique : un ensemble de centrales (avec cout,
capacites, facteur d'emission) et une demande horaire a satisfaire, avec une production
renouvelable stochastique.


In [1]:
# Installation des dependances (executees dans les cycles suivants avec les solveurs)
# %pip install ortools numpy scipy  # decommenter pour execution complete

from dataclasses import dataclass, field
from typing import List, Dict
import numpy as np


@dataclass
class PowerPlant:
    """Centrale de production electrique."""
    name: str
    pmin: float          # puissance minimale stable (MW)
    pmax: float          # puissance maximale (MW)
    cost_per_mw: float   # cout variable (EUR/MWh)
    co2_per_mw: float    # emission (kg/MWh)
    is_renewable: bool = False


@dataclass
class PowerNetwork:
    """Jumeau numerique d'un reseau electrique horaire."""
    plants: List[PowerPlant]
    demand_mw: List[float]               # demande par heure (MW)
    renewable_forecast_mean: List[float]  # production renouvelable moyenne attendue (MW)
    renewable_forecast_std: List[float]   # incertitude (ecart-type, MW)

    def total_capacity(self) -> float:
        return sum(p.pmax for p in self.plants)

    def net_demand(self, hour: int) -> float:
        """Demande a couvrir par les centrales pilotables apres renouvelable."""
        return max(0.0, self.demand_mw[hour] - self.renewable_forecast_mean[hour])


def demo_network() -> PowerNetwork:
    """Reseau de demonstration : 3 centrales pilotables + renouvelable sur 6 heures."""
    plants = [
        PowerPlant('Charbon', 50, 400, cost_per_mw=30, co2_per_mw=900),
        PowerPlant('Gaz',    20, 250, cost_per_mw=60, co2_per_mw=400),
        PowerPlant('Hydro',   0, 150, cost_per_mw=10, co2_per_mw=0,  is_renewable=False),
    ]
    return PowerNetwork(
        plants=plants,
        demand_mw=[500, 520, 480, 540, 600, 580],
        renewable_forecast_mean=[80, 60, 40, 50, 120, 150],
        renewable_forecast_std=[15, 12, 10, 12, 25, 30],
    )


net = demo_network()
print(f'Reseau : {len(net.plants)} centrales, capacite totale {net.total_capacity()} MW')
for h in range(len(net.demand_mw)):
    print(f'  H{h}: demande {net.demand_mw[h]} MW, renouvelable {net.renewable_forecast_mean[h]} '
          f'+/- {net.renewable_forecast_std[h]}, demande nette {net.net_demand(h):.1f} MW')


Reseau : 3 centrales, capacite totale 800 MW
  H0: demande 500 MW, renouvelable 80 +/- 15, demande nette 420.0 MW
  H1: demande 520 MW, renouvelable 60 +/- 12, demande nette 460.0 MW
  H2: demande 480 MW, renouvelable 40 +/- 10, demande nette 440.0 MW
  H3: demande 540 MW, renouvelable 50 +/- 12, demande nette 490.0 MW
  H4: demande 600 MW, renouvelable 120 +/- 25, demande nette 480.0 MW
  H5: demande 580 MW, renouvelable 150 +/- 30, demande nette 430.0 MW


## Partie 1 : Le Dispatcher sous Contraintes (OR-Tools CP-SAT)

**Theorie** : le *unit commitment* est un probleme NP-difficile. On modelise, pour chaque
heure et chaque centrale pilotable, une variable binaire (activee) et une variable continue
(niveau de production), soumises aux contraintes :

- somme des productions = demande nette (equilibre offre/demande) ;
- `pmin <= production <= pmax` quand la centrale est activee ;
- reserve tournante : capacite excédentaire disponible >= marge de securite.

L'objectif primaire : minimiser le cout total.


In [2]:
# Partie 1 : modele CP-SAT du dispatch (unit commitment)
# TODO etudiant : implementer solve_dispatch_cp avec ortools.sat.python.cp_model.
# Pour chaque heure et centrale pilotable : binaire `on` + entier `p`.
# Contraintes : equilibre offre/demande nette + bornes pmin <= p <= pmax.
# Objectif : minimiser le cout total.
def solve_dispatch_cp(network, cost_weights=None):
    """Resout le unit commitment par programmation par contraintes (CP-SAT)."""
    result = None  # TODO etudiant
    return result


dispatch = solve_dispatch_cp(net)
print('Dispatch CP-SAT :', dispatch)


Dispatch CP-SAT : None


### Exercice 1 : Reserve tournante (securite n-1)

Ajoutez au modele CP-SAT une contrainte de **reserve tournante** : a chaque heure, la somme
des capacites disponibles (non utilisees) des centrales activees doit couvrir la perte de la
plus grande centrale en service (critere n-1).

**Indice** : pour chaque heure, la reserve = `sum(pmax_i - prod_i pour i active)` doit etre
`>= max(pmax_i pour i active)`.


In [3]:
# Exercice 1 : contrainte de reserve tournante n-1
def solve_dispatch_with_spinning_reserve(network: PowerNetwork):
    """Dispatch CP-SAT avec reserve tournante n-1."""
    result = None  # TODO etudiant : etendre solve_dispatch_cp avec la reserve n-1
    return result


## Partie 2 : Le Previsionniste Probabiliste (modele bayesien)

**Theorie** : la production renouvelable est aleatoire. On modelise l'ecart
`demande - renouvelable` comme une variable aleatoire et on infere sa distribution, pour
quantifier le **risque de defaillance** (probabilite que la demande nette depasse la capacite
pilotable disponible).


In [4]:
# Partie 2 : modele bayesien de l'incertitude renouvelable
# TODO etudiant : P(demande nette reelle > capacite pilotable).
# Modele l'ecart offre/demande comme une gaussienne et utilise sa survivor function.
from math import erfc, sqrt


def failure_risk(network, hour):
    """Probabilite que la demande nette (aleatoire) depasse la capacite pilotable."""
    risk = None  # TODO etudiant : calculer via la survivor function d'une gaussienne
    return risk


print('Risque de defaillance (a completer par l' + 'etudiant) :')
for h in range(len(net.demand_mw)):
    print(f'  H{h}: risque = {failure_risk(net, h)}')


Risque de defaillance (a completer par letudiant) :
  H0: risque = None
  H1: risque = None
  H2: risque = None
  H3: risque = None
  H4: risque = None
  H5: risque = None


### Exercice 2 : Sensibilite au prior sur la variabilite eolienne

Analysez comment le risque de defaillance (Partie 2) evolue quand on modifie l'incertitude
du renouvelable (multipliez `renewable_forecast_std` par 0.5, 1.0, 2.0). Quelle heure
devient la plus risqueuse ? Concluez sur la robustesse du dispatch determine en Partie 1.


In [5]:
# Exercice 2 : analyse de sensibilite au prior d'incertitude
def sensitivity_to_renewable_std(network: PowerNetwork, factors=(0.5, 1.0, 2.0)):
    """Retourne le risque max par heure pour chaque facteur d'incertitude."""
    result = None  # TODO etudiant : boucler sur les facteurs, retourner {factor: max_risk}
    return result


## Partie 3 : L'Optimiseur Multi-Objectif

**Theorie** : on veut minimiser simultanement le **cout** economique, les **emissions** CO2
et le **risque** de defaillance. Ces objectifs sont conflictuels (le charbon est bon marche
mais tres emissif). On construit un score pondere `S = a*cout + b*co2 + c*risque` et on
cherche le dispatch minimisant S, en explorant le front de Pareto.


In [6]:
# Partie 3 : score multi-objectif (cout + CO2 + risque)
# TODO etudiant : normaliser chaque objectif puis combiner par somme ponderee.
def dispatch_metrics(dispatch, network):
    """Calcule cout total, CO2 total et risque max d'un dispatch."""
    if dispatch is None:
        return {'cost': float('inf'), 'co2': float('inf'), 'max_risk': 1.0}
    metrics = None  # TODO etudiant : boucler sur heures + plants
    return metrics


def multi_objective_score(dispatch, network, weights=(1.0, 1.0, 1.0), norms=None):
    """Score = w_c*cout_norm + w_co2*co2_norm + w_r*risque_norm."""
    score = None  # TODO etudiant
    return score


print('Score multi-objectif :', multi_objective_score(dispatch, net))


Score multi-objectif : None


### Exercice 3 : Ajouter un troisieme objectif (equite territoriale)

Ajoutez un troisieme objectif : **equite territoriale** (eviter qu'une meme region subisse
toutes les centrales les plus polluantes). Supposons que chaque centrale a un attribut
`region`. Mesurez la concentration regionale de la pollution et ajoutez-la au score.


In [7]:
# Exercice 3 : objectif d'equite territoriale
def territorial_equity_score(dispatch, network: PowerNetwork) -> float:
    """Mesure de concentration de la pollution par region (0 = equitable)."""
    score = None  # TODO etudiant : mesurer la concentration (ex: variance/entropy regionale des co2)
    return score


## Partie 4 : Decision finale et analyse comparative

**Synthese** : on compare les strategies (CP-SAT pur, multi-objectif weighted-sum, front de
Pareto) sur les 3 axes cout/CO2/fiabilite + l'equite territoriale. Le tableau comparatif et
le graphique radar seront produits une fois les Parties 1-3 executees.


In [8]:
# Partie 4 : analyse comparative des strategies de dispatch
# TODO etudiant : comparer 3 strategies (min cout / min CO2 / compromis).
def comparative_analysis(network):
    """Compare 3 strategies CP-SAT sur cout / CO2 / risque."""
    analysis = None  # TODO etudiant : re-solve 3x, normaliser, scorer
    return analysis


print('Analyse comparative :', comparative_analysis(net))


Analyse comparative : None


## Conclusion

Cette etude de cas illustre la **composition ordonnee** des paradigmes IA sur un probleme
metier reel (transition energetique) : on filtre les dispatches impossibles (CP-SAT) avant
de modeliser l'incertitude (bayesien) avant d'optimiser (multi-objectif). Inverser l'ordre
produit un systeme soit trop rigide (contraintes ignorent l'aleatoire), soit trop flou
(decision sans contraintes physiques).

**Etat de la fondation** : jumeau numerique operationnel (couche 0), 4 couches de solveurs
modelisees en stub. Implementations + execution + analyse comparative : cycles suivants.
